# Phase 3 - Baseline forecasts

Thin notebook that orchestrates `seercast.training.train_baselines` to:

1. Load the joined base table for `CA_1`.
2. Run all four baselines (naive, seasonal naive, MA-28, seasonal MA) through the rolling-origin backtester.
3. Persist predictions/scores/summary to `outputs/reports/`.
4. Render a few diagnostic plots: WAPE by horizon per model, predicted vs actual for one example series.

All real logic lives in `seercast.models.baselines`, `seercast.evaluation.backtesting`, and `seercast.evaluation.metrics`. This notebook is a runnable demo and a place to look at the results.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.config import ARTIFACTS
from seercast.training.train_baselines import run as run_baselines

REPO_ROOT

## 1. Run the backtest

In [ ]:
result = run_baselines()
predictions = result['predictions']
scores = result['scores']
summary = result['summary']
predictions.head()

## 2. Overall ranking (WAPE is primary)

In [ ]:
summary

## 3. WAPE by horizon, one line per model

Healthy baselines should have roughly flat WAPE across horizons (since they don't try to model trend / dynamics). The seasonal baselines should beat the non-seasonal ones at every horizon if there is meaningful weekly seasonality.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for name, sub in scores.groupby('model'):
    ax.plot(sub['horizon'], sub['WAPE'], marker='o', label=name)
ax.set_xlabel('horizon (days ahead)')
ax.set_ylabel('WAPE')
ax.set_title('Baseline WAPE by horizon (CA_1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Eyeball one series: actual vs forecast at the latest origin

Pick the highest-volume id and the latest backtest origin. For any baseline this should look reasonable; if it doesn't, the backtest plumbing is wrong.

In [ ]:
latest_origin = predictions['origin_date'].max()
top_id = (
    predictions[predictions['origin_date'] == latest_origin]
    .groupby('id')['actual'].sum().idxmax()
)
print('eyeballing id:', top_id, 'origin:', latest_origin)

sub = predictions[(predictions['origin_date'] == latest_origin) & (predictions['id'] == top_id)]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sub[sub['model'] == 'naive']['target_date'], sub[sub['model'] == 'naive']['actual'],
        marker='.', color='black', label='actual')
for name, ssub in sub.groupby('model'):
    ax.plot(ssub['target_date'], ssub['prediction'], marker='o', linestyle='--', label=name)
ax.set_title(f'{top_id} - actual vs baseline forecasts (origin {latest_origin.date()})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Next:** Phase 4 - leakage-safe feature engineering for the LightGBM model.